# UNO Vision — Project Report

This notebook describes the full pipeline for detecting and classifying UNO cards.
The input is a top-down photo of an ongoing game; the output is a CSV describing the
cards visible for each player and the center pile.

**Pipeline overview**

```
Raw image
    │
    ├── Background removal (src/background_removal.py)
    ├── Region splitting into player areas / center
    │
    └── For each region:
            │
            ├── 1. HSV segmentation  ──► color masks
            ├── 2. Quad detection    ──► card rectangles
            ├── 3. Image extraction  ──► 140×218 images
            └── 4. Corner matching   ──► label (e.g. r_5, b_skip)
```

## 1. Card Detection (`src/card_detection.py`)

Detection transforms a BGR region into a list of `(card_image, color)` pairs.
It consists of three stages: mask preprocessing, quadrilateral detection, and image extraction.

### 1.1 HSV Segmentation — `preprocess_mask`

UNO cards stand out from the background by their vivid color. We work in HSV space
rather than BGR because the hue channel H is robust to lighting variations.

Six color ranges are defined:

| Color | Hue (H) | Note |
|-------|---------|------|
| Red lo | 0 – 8° | Red wraps around H = 0 in HSV |
| Red hi | 165 – 179° | Complement of red |
| Yellow | 22 – 36° | |
| Green | 45 – 85° | |
| Blue | 90 – 130° | |
| Dark (wild/+4) | S ≤ 100, V ≤ 120 | Black body of joker cards |

The saturation `sat_lo` and value `val_lo` lower bounds for colored ranges are tunable
parameters to adapt to lighting conditions.

**Morphological pipeline applied to each mask:**
1. `cv2.inRange` → raw binary mask
2. `MORPH_OPEN` (kernel ≈ 1.7 % of region size) → removes small noise blobs
3. `MORPH_CLOSE` (kernel ≈ 10 % of region size) → fills internal gaps inside the card
4. `_fill_holes`: flood-fill from the image border closes open contours

**Suppression of colored wedges inside joker cards:**  
Wild/+4 cards have a dark body but colored sectors at the center.
After hole-filling, the union of all dark masks covers the full joker body.
This filled dark mask is subtracted from every colored mask
(`bitwise_and(colored, NOT(dark))`) so the wedges no longer produce false blobs
in the colored masks.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, ".")

import cv2
import matplotlib.pyplot as plt
from src.card_detection import preprocess_mask, debug_mask

DATA_DIR = "iapr-26-uno-vision-challenge/train_images"
IMAGE_ID = "L1000770"

img = cv2.imread(os.path.join(DATA_DIR, IMAGE_ID + ".jpg"))
H, W = img.shape[:2]

# Center region (middle third in both axes)
region = img[H//3 : 2*H//3, W//3 : 2*W//3]

mask_vis = debug_mask(region, name="center", sat_lo=80, val_lo=120)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(region, cv2.COLOR_BGR2RGB))
axes[0].set_title("Center region")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(mask_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title("Color masks (red=red, cyan=yellow, green=green, blue=blue, gray=dark)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### 1.2 Quad Detection — `get_quads`

From the masks, we search for rectangles corresponding to cards.
Three complementary methods are used in order of decreasing reliability.

**Pre-processing:** red-lo and red-hi masks are merged (`bitwise_or`) before processing.
This prevents the same red card from being detected twice (once per range).

---

#### Method 1 — Direct detection (M1)

For each contour in a mask:
- `cv2.minAreaRect` → minimum-area oriented rectangle
- Criterion: **area ≥ threshold** (15 % of the minimum region dimension)² AND **side ratio** within ±30 % of the UNO card ratio (87/56 ≈ 1.55)
- If both criteria are satisfied → the rectangle is directly accepted as a quad

This method works for fully visible cards with a compact mask.

---

#### Method 2 — Fragment pairing (M2)

When a card is partially hidden or its mask splits into multiple blobs, M2 tries to
**combine fragments of the same color**:

- Colored cards: all **pairs** of same-color blobs are tested
- Dark cards: **all subsets** of dark blobs are tested (they can fragment into 3+)
- For each combination: `minAreaRect` on the union of points → aspect ratio check
- Score: distance to the reference area (derived from M1 cards already found)
- Greedy selection: each blob is used at most once

---

#### Method 3 — Edge-based reconstruction (M3)

For orphan blobs (not claimed by M1 or M2), the quadrilateral is reconstructed from a
**visible 90° corner on the blob's convex hull**:

1. `cv2.convexHull` + `approxPolyDP` on the blob points
2. For each consecutive triplet of hull vertices (A, B, C) with an angle at B ≈ 90°:
   - Unit vector `u = (A − B) / |A − B|` (direction of edge BA)
   - Exact perpendicular `v` oriented toward C (±90° rotation of u)
   - Optional confirmation: search for a 3rd hull edge parallel to u or v
     at distance ≈ card width or height → disambiguates which of u, v is the width
   - Fixed-size card rectangle (scaled from 140 × 218) anchored at corner B
3. Acceptance: ≥ 90 % of the blob covered by the proposed quad
4. Score: minimum mean distance from quad edges to the blob boundary
5. Deduplication by IoU > 50 %

The fixed card size is extrapolated from the median area of M1+M2 candidates
(`ref_w = sqrt(ref_area / aspect_ratio)`).

In [ ]:
from src.card_detection import debug_quads

vis = debug_quads(region, sat_lo=80, val_lo=120)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Detected quads — M1 (direct), M2 (pairing), M3 (reconstruction)")
plt.axis("off")
plt.tight_layout()
plt.show()

### 1.3 Image Extraction — `get_card_images`

For each candidate quad:

1. **Coverage filter** (colored cards only): at least 60 % of the quad's interior must be
   covered by detected pixels (union of all masks). This rejects reconstructed quads
   that fall mostly on background.

2. **Perspective transform** (`_warp_card`):
   - The quad is **expanded by 25 %** from its centroid to capture the card borders
   - `cv2.getPerspectiveTransform` + `cv2.warpPerspective` → rectified image
   - If width > height, rotate 90° to enforce portrait orientation
   - **Tight crop** (skipped for dark cards): detects colored pixels (S > 25 or V > 180)
     within the inner 7.5 % margin to locate the actual card edge and trim white padding
   - Final resize: **140 × 218 pixels** (canonical format)

3. **Size filter**: if ≥ 2 candidates, quads whose area deviates more than 50 % from the
   reference (`_CARD_REF_AREA_PX = 140 000 px`) are rejected.
   Tolerance is doubled for dark cards (fragmented blobs → noisier area estimates).

4. **Non-Maximum Suppression (NMS)**: sort by bounding-box area descending;
   for each quad, suppress if IoU > 50 % with an already-kept quad **of the same color**.
   Additionally, any colored quad whose centroid falls inside a kept dark quad is suppressed
   (prevents a joker corner from being detected as a colored card).

In [ ]:
from src.card_detection import detect_cards

cards = detect_cards(region, sat_lo=80, val_lo=120)
print(f"{len(cards)} card(s) detected")

if cards:
    fig, axes = plt.subplots(1, len(cards), figsize=(4 * len(cards), 5))
    if len(cards) == 1:
        axes = [axes]
    for ax, (card_img, color) in zip(axes, cards):
        ax.imshow(cv2.cvtColor(card_img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"color: {color}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2. Card Classification (`src/card_classification.py`)

Once each card is extracted as a 140 × 218 image, we identify its **value**
(0–9, skip, reverse, draw_2, wild, draw_4) to form the full label (e.g. `r_5`, `b_skip`).

The method relies on **corner patch matching**: each UNO card prints its value
as white ink symbols on the colored background in the top-left and bottom-right corners.

### 2.1 Corner Extraction — `_extract_corner`

Each UNO card displays its value in the top-left (TL) corner and in the bottom-right
(BR) corner upside-down. Both corners are exploited to handle partially hidden cards.

**TL corner**: patch `img[7:55, 7:40]` → size **48 × 33 px**  
**BR corner**: patch `img[H-55:H-7, W-40:W-7]`, rotated 180° to align its orientation with TL

The 7-pixel white card border is skipped (`_BORDER = 7`) to retain only the printed symbol.

**Binarization**: a pixel is considered "white" if all its BGR channels exceed 210
(`_WHITE_THRESH = 210`). This produces a float32 binary mask (1.0 = white ink, 0.0 = colored
background), capturing the symbol independently of the background color.

### 2.2 Template Building — `build_templates_from_labeled`

Templates are built from images manually labeled using `label_cards.py`.

Directory layout:
```
labeled_cards/
    0/        ← all card images with value 0
    1/
    ...
    skip/
    reverse/
    draw_2/
    wild/
    draw_4/
```

For each value:
1. Each image is resized to 140 × 218
2. Both the TL mask **and** the BR mask are extracted (two observations per image)
3. The final template is the **mean** of all masks, equally weighted

Averaging smooths out lighting and printing variations between images.
The resulting template looks like a blurred version of the printed corner symbol.

In [ ]:
import numpy as np
from src.card_classification import load_templates

templates = load_templates("templates")
print(f"Available values: {sorted(templates.keys())}")

values = sorted(templates.keys())
n = len(values)
fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
for ax, val in zip(axes, values):
    ax.imshow(templates[val], cmap="gray", vmin=0, vmax=1)
    ax.set_title(val, fontsize=9)
    ax.axis("off")
fig.suptitle("Corner templates (mean white-ink mask over all labeled images)", fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 Classification — `classify_card`

Classification uses `cv2.matchTemplate` with the **TM_CCOEFF_NORMED** metric
(normalized cross-correlation), which returns a score in [-1, 1] robust to
global brightness variations.

Since the template and the patch have the same size (48 × 33), `matchTemplate` returns
a (1, 1) array; the scalar score is `result[0, 0]`.

**Dark cards** (`color == 'dark'`, wild / +4):
- Only the `wild` and `draw_4` templates compete
- If both are absent: density heuristic on white pixels in TL
  (draw_4 prints "+4" → denser than the wild circle)

**Colored cards**:
- Both TL **and** BR patches are extracted from the detected card
- For each candidate value:
  `score(v) = max( matchTemplate(TL, tmpl[v]), matchTemplate(BR, tmpl[v]) )`
- The winning value is the argmax of the scores
- The final label is `{color_prefix}_{value}`:
  `r_5`, `y_skip`, `g_draw_2`, `b_reverse`, `wild`, `draw_4`

Taking the max over TL and BR allows correct classification of partially hidden cards:
if TL is obstructed, BR provides a reliable score.

In [ ]:
from src.card_classification import classify_card

cards = detect_cards(region, sat_lo=80, val_lo=120)
print(f"{len(cards)} card(s) detected")

if cards:
    fig, axes = plt.subplots(1, len(cards), figsize=(4 * len(cards), 5))
    if len(cards) == 1:
        axes = [axes]
    for ax, (card_img, color) in zip(axes, cards):
        label = classify_card(card_img, color, templates)
        ax.imshow(cv2.cvtColor(card_img, cv2.COLOR_BGR2RGB))
        ax.set_title(label, fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### 2.4 Global Evaluation

The script `test_card_classification.py` evaluates classification on the full training dataset
and reports accuracy per region (players 1–4 and center):
- **count accuracy**: fraction of images where the number of detected cards is correct
- **label accuracy**: fraction of images where both the count and the labels are correct (after sorting)

For failing images, a figure is shown with the region, the mask, and the detected cards
annotated with their predicted labels.

In [ ]:
# To run the full evaluation (shows only failures):
# !python test_card_classification.py

# For a specific image:
# !python test_card_classification.py L1000770